# 🛣️ Fase 1 — Exploración del Dataset
## Detección de Baches y Anomalías Viales
**Universidad Andina del Cusco — Inteligencia Artificial Aplicada 2026**

---
**Equipo:**
- Orellana Cusihuaman Luis Anthony — Preprocesamiento
- Montufar Diaz Alfredo Gerardo — Modelado
- Vilca Ramos Luis Gerardo — Métricas y Despliegue

**Objetivo de este notebook:** Explorar el dataset, verificar la distribución de clases, visualizar ejemplos y aplicar el preprocesamiento inicial.

## 0. Instalación de dependencias (solo ejecutar en Google Colab)

In [ ]:
# Instalar dependencias (ejecutar solo en Colab)
# !pip install -r requirements.txt

# Clonar el repo si estás en Colab
# !git clone https://github.com/AlfredoMonT/IABACHES.git
# %cd IABACHES

## 1. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
import sys
sys.path.append('..')

from src.utils.helpers import set_seed, print_separator
from src.preprocessing.image_processor import preprocess_image, load_image

# Semilla para reproducibilidad
set_seed(42)

# Rutas del dataset
DATA_RAW   = Path('../data/raw')
DATA_PROC  = Path('../data/processed')

CLASSES = ['pothole', 'normal']
print('✅ Imports completados')

## 2. Descarga del Dataset desde Kaggle

In [ ]:
# Configurar Kaggle API (subir kaggle.json a Colab)
# from google.colab import files
# files.upload()  # Sube tu kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Descargar dataset
# !kaggle datasets download -d muskanverma24/pothole-detection-dataset-yolov11-optimized -p ../data/raw --unzip
print('⬇️  Ejecutar celdas de descarga cuando tengas el kaggle.json configurado')

## 3. Análisis Exploratorio — Distribución de Clases

In [ ]:
# Contar imágenes por clase
from src.utils.helpers import count_images_per_class

counts = count_images_per_class(str(DATA_RAW), CLASSES)
total  = sum(counts.values())

print(f'📊 Distribución del Dataset:')
for cls, n in counts.items():
    pct = n / total * 100 if total > 0 else 0
    print(f'   {cls:10s}: {n:5d} imágenes ({pct:.1f}%)')
print(f'   {"TOTAL":10s}: {total:5d} imágenes')

# Gráfico de distribución
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.keys(), counts.values(),
              color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
ax.set_title('Distribución de Clases — Dataset de Baches', fontsize=14)
ax.set_ylabel('Número de Imágenes')
for bar, (cls, n) in zip(bars, counts.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{n}\n({n/total*100:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/distribucion_clases.png', dpi=150)
plt.show()

## 4. Visualización de Ejemplos del Dataset

In [ ]:
# Mostrar ejemplos de cada clase: original vs preprocesada
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Ejemplos del Dataset — Original vs Preprocesada (HOG-ready)', fontsize=14)

for row, cls in enumerate(CLASSES):
    cls_dir = DATA_RAW / cls
    images  = list(cls_dir.glob('*.jpg'))[:4] if cls_dir.exists() else []

    for col, img_path in enumerate(images[:4]):
        original = cv2.imread(str(img_path))
        original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
        processed = preprocess_image(str(img_path), apply_contrast=True)

        if col < 2:
            axes[row][col].imshow(original)
            axes[row][col].set_title(f'{cls.upper()} — Original', fontsize=9)
        else:
            axes[row][col].imshow(processed, cmap='gray')
            axes[row][col].set_title(f'{cls.upper()} — Preprocesada', fontsize=9)
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/ejemplos_dataset.png', dpi=150)
plt.show()
print('✅ Visualización completada')

## 5. Siguiente Paso
➡️ Continuar en: **`fase1_extraccion_hog.ipynb`** para extraer los vectores HOG y construir la matriz de características.